In [14]:
from clear_text import clean_text
from data_loader import load_pres, load_movies
from features import vectorize_and_split
from models import train_logistic_regression, train_naive_bayes
from evaluation import evaluate_model
from sklearn import svm
from sklearn.model_selection import train_test_split

— Chargement des données

In [2]:
# Chemins vers les datasets
pres_path = "./Dataset/corpus.tache1.learn.utf8"
movies_path = "./Dataset/movies1000/"

# Chargement Président (Chirac / Mitterrand)
alltxts, alllabs = load_pres(pres_path)

# Chargement Films (positif / négatif)
alltxts_f, alllabs_f = load_movies(movies_path)

# Vérification
print("Nombre de textes président :", len(alltxts))
print("Nombre de textes films :", len(alltxts_f))

Nombre de textes président : 57413
Nombre de textes films : 2000


— Nettoyage des textes

In [3]:
# Président
texts_clean_pres = [clean_text(t) for t in alltxts]

# Films
texts_clean_movies = [clean_text(t) for t in alltxts_f]

# Vérification
print("Exemple texte président avant nettoyage :")
print(alltxts[0])

print("\nExemple texte président après nettoyage :")
print(texts_clean_pres[0])

print("\nExemple texte film avant nettoyage :")
print(alltxts_f[0][:300])

print("\nExemple texte film après nettoyage :")
print(texts_clean_movies[0][:300])

Exemple texte président avant nettoyage :
> Quand je dis chers amis, il ne s'agit pas là d'une formule diplomatique, mais de l'expression de ce que je ressens.

Exemple texte président après nettoyage :
quand di cher ami agit formul diplomatiqu express ressen

Exemple texte film avant nettoyage :
assume nothing . 
the phrase is perhaps one of the most used of the 1990's , as first impressions and rumors are hardly ever what they seem to be . 
the phrase especially goes for oscar novak , an architect who is the main focus of three to tango , a delightful , funny romantic comedy about assumpti

Exemple texte film après nettoyage :
assum noth phrase perhap one use first impress rumor hardli ever seem phrase especi goe oscar novak architect main focu three tango delight funni romant comedi assumpt novak matthew perri shi clumsi chicago base architect along openli gay partner peter steinberg oliv platt fight project day day one 


- Extraction du vocabulaire (BoW)


In [15]:
from features import (
    vectorize_bow,
    vectorize_bow_binary,
    vectorize_tfidf,
    vectorize_ngrams,
    vectorize_limited,
    get_vocab_size,
    get_top_words
)


In [12]:
#Tester la taille du vocabulaire + top mots 

# Président
X_bow_pres, vec_bow_pres = vectorize_bow(texts_clean_pres)

print("========== PRÉSIDENT ==========")
print("Taille du vocabulaire (président) :", get_vocab_size(vec_bow_pres))
print("Top 20 mots fréquents (président) :")
print(get_top_words(vec_bow_pres, X_bow_pres, 20))


========== PRÉSIDENT ==========
Taille du vocabulaire (président) : 17891
Top 20 mots fréquents (président) :
[('plu', np.int64(7445)), ('tout', np.int64(6461)), ('franc', np.int64(5294)), ('cett', np.int64(4883)), ('pay', np.int64(4494)), ('aussi', np.int64(4366)), ('etr', np.int64(3853)), ('nom', np.int64(3578)), ('grand', np.int64(3470)), ('tou', np.int64(3457)), ('meme', np.int64(3428)), ('bien', np.int64(3026)), ('comm', np.int64(2812)), ('entr', np.int64(2698)), ('etat', np.int64(2682)), ('ete', np.int64(2628)), ('europ', np.int64(2611)), ('hui', np.int64(2582)), ('aujourd', np.int64(2581)), ('autr', np.int64(2534))]


— TF-IDF + split

In [5]:

# Président
vectorizer_pres, X_train_pres, X_test_pres, y_train_pres, y_test_pres = vectorize_and_split(
    texts_clean_pres,
    alllabs,
    test_size=0.2,
    random_state=42,
    ngram_range=(1, 2),
    max_features=10000
)

# Films
vectorizer_movies, X_train_movies, X_test_movies, y_train_movies, y_test_movies = vectorize_and_split(
    texts_clean_movies,
    alllabs_f,
    test_size=0.2,
    random_state=42,
    ngram_range=(1, 1),
    max_features=None
)

# Vérification
print("Président train :", X_train_pres.shape)
print("Président test  :", X_test_pres.shape)

print("Films train :", X_train_movies.shape)
print("Films test  :", X_test_movies.shape)

Président train : (45930, 10000)
Président test  : (11483, 10000)
Films train : (1600, 23476)
Films test  : (400, 23476)


- LOGISTIC REGRESSION

In [6]:

# entrainement 

# Président : classes déséquilibrées → balanced=True
model_pres = train_logistic_regression(
    X_train_pres,
    y_train_pres,
    balanced=True
)

# Films : baseline standard
model_movies = train_logistic_regression(
    X_train_movies,
    y_train_movies,
    balanced=False
)

# evaluation sur test

print("========== Regression Logistic =========")
# Évaluation Président
print("\n===== PRÉSIDENT =====")
results_pres = evaluate_model(model_pres, X_test_pres, y_test_pres)

# Évaluation Films
print("\n===== FILMS =====")
results_movies = evaluate_model(model_movies, X_test_movies, y_test_movies)

========== Regression Logistic =========

===== PRÉSIDENT =====
Accuracy  : 0.8265
Precision : 0.9519
Recall    : 0.8430
F1-score  : 0.8941

===== FILMS =====
Accuracy  : 0.8275
Precision : 0.8164
Recall    : 0.8450
F1-score  : 0.8305


- NAIVE BAYES

In [7]:
# entrainement

# Président
model_nb_pres = train_naive_bayes(X_train_pres, y_train_pres)

# Films
model_nb_movies = train_naive_bayes(X_train_movies, y_train_movies)

# evaluation sur test
print("========== NAIVE BAYES ==========")

print("\n===== PRÉSIDENT =====")
evaluate_model(model_nb_pres, X_test_pres, y_test_pres)

print("\n===== FILMS =====")
evaluate_model(model_nb_movies, X_test_movies, y_test_movies);

========== NAIVE BAYES ==========

===== PRÉSIDENT =====
Accuracy  : 0.8805
Precision : 0.8804
Recall    : 0.9981
F1-score  : 0.9356

===== FILMS =====
Accuracy  : 0.8075
Precision : 0.8220
Recall    : 0.7850
F1-score  : 0.8031


- SVM

In [9]:
#entrainement

# Président
model_svm_pres = train_svm(X_train_pres, y_train_pres)

# Films
model_svm_movies = train_svm(X_train_movies, y_train_movies)

# evaluation sur test
print("========== SVM ==========")
print("\n===== PRÉSIDENT =====")
evaluate_model(model_svm_pres, X_test_pres, y_test_pres)

print("\n===== FILMS =====")
evaluate_model(model_svm_movies, X_test_movies, y_test_movies);


========== SVM ==========

===== PRÉSIDENT =====
Accuracy  : 0.8959
Precision : 0.9167
Recall    : 0.9682
F1-score  : 0.9418

===== FILMS =====
Accuracy  : 0.8375
Precision : 0.8261
Recall    : 0.8550
F1-score  : 0.8403
